## 🔗 Live Deployment Link: https://YOUR-APP-NAME.streamlit.app
*(replace the link above with your real Streamlit Cloud link after you deploy)*

# Predicting How Many Olympics an Athlete Will Take Part In
### Comparing 5 ML algorithms with 10-Fold Cross-Validation

**Course:** Machine Learning Fundamentals (AIC354) | **Assignment 3, Task 2**

**Problem:** Predict `games_participations`, the number of Olympic Games an athlete took part in (from 1 to 10). This is a regression problem, like the Car Price Prediction System.

**Dataset:** `olympic_athletes.csv` (75,904 Olympic athletes, scraped from olympics.com). I use the **full dataset**.

**Algorithms (none of them are used in the car price notebooks):**
1. Elastic Net
2. Bayesian Ridge
3. Huber Regressor
4. Bagging Regressor
5. MLP Regressor (neural network)

**Metric:** RMSE (Root Mean Square Error), lower is better.

## Step 1: Import libraries

In [ ]:
import os
import pickle
import re
import time

import numpy as np
import pandas as pd

import sklearn
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# The 5 regression algorithms (different from the car price notebooks)
from sklearn.linear_model import ElasticNet, BayesianRidge, HuberRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

import matplotlib.pyplot as plt
from tqdm import tqdm

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("scikit-learn version:", sklearn.__version__)
print("(Write this version in requirements.txt of the web app so the model loads without errors)")

## Step 2: Upload the dataset (Google Colab)

In [ ]:
DATA_FILE = "olympic_athletes.csv"

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(DATA_FILE):
    print(f"Click 'Choose Files' and select '{DATA_FILE}' ...")
    uploaded = files.upload()
    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != DATA_FILE:
        os.rename(uploaded_name, DATA_FILE)

if os.path.exists(DATA_FILE):
    print(f"'{DATA_FILE}' is ready.")
else:
    raise FileNotFoundError(f"'{DATA_FILE}' not found. Run this cell again and upload it.")

## Step 3: Load and explore the data

In [ ]:
raw = pd.read_csv(DATA_FILE)
print("Shape (rows, columns):", raw.shape)
raw.head()

In [ ]:
print("Missing values per column:")
print(raw.isna().sum())
print()
print("How many Games did athletes take part in?")
print(raw["games_participations"].value_counts().sort_index())

## Step 4: Data cleaning and feature creation

The raw file has mostly text columns, so I create numeric features from them:

* `first_game_year`: the year taken from the `first_game` text (for example "Rio 2016" gives 2016)
* `season`: Winter or Summer (taken from the `first_game` text)
* `birth_year`: from `athlete_year_birth`
* `age_at_first_game`: first game year minus birth year
* `gold`, `silver`, `bronze`: medal counts read from the messy `athlete_medals` text (G, S, B). Athletes with no medals get 0.
* `has_bio`: 1 if the athlete has a biography, otherwise 0

The columns `athlete_url`, `athlete_full_name` and `bio` are identifiers or free text, so they are not used directly.
Missing birth years and first game years are filled inside the pipeline, so **no rows are deleted** and the full dataset is used.

In [ ]:
df = raw.copy()

# year and season from text like "Beijing 2022" / "Lillehammer 1994 Winter"
df["first_game_year"] = df["first_game"].str.extract(r"(\d{4})")[0].astype(float)
df["season"] = np.where(df["first_game"].fillna("").str.contains("Winter"), "Winter", "Summer")

df["birth_year"] = df["athlete_year_birth"]
df["age_at_first_game"] = df["first_game_year"] - df["birth_year"]

# medal text looks like "\n\n\n1\n\nG\n\n\n\n2\n\nB\n\n"  (number then letter)
def count_medals(text, letter):
    if not isinstance(text, str):
        return 0
    found = re.findall(r"(\d+)\s*" + letter + r"(?![A-Za-z])", text)
    return sum(int(n) for n in found)

df["gold"] = df["athlete_medals"].apply(lambda t: count_medals(t, "G"))
df["silver"] = df["athlete_medals"].apply(lambda t: count_medals(t, "S"))
df["bronze"] = df["athlete_medals"].apply(lambda t: count_medals(t, "B"))

df["has_bio"] = (df["bio"].fillna("").str.strip().str.len() > 5).astype(int)

categorical_cols = ["season"]
numeric_cols = ["first_game_year", "birth_year", "age_at_first_game", "gold", "silver", "bronze", "has_bio"]
target_col = "games_participations"

X = df[categorical_cols + numeric_cols]
y = df[target_col].values

print("Rows used:", len(X))
print("Total medals found -> gold:", df["gold"].sum(), "| silver:", df["silver"].sum(), "| bronze:", df["bronze"].sum())
X.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
raw["games_participations"].value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#4C78A8")
ax[0].set_title("Number of athletes by Games participations")
ax[0].set_xlabel("Games participations")
ax[0].set_ylabel("Athletes")

df.groupby(df["gold"] + df["silver"] + df["bronze"] > 0)["games_participations"].mean().plot(
    kind="bar", ax=ax[1], color="#F58518")
ax[1].set_xticklabels(["No medal", "Medalist"], rotation=0)
ax[1].set_title("Average participations: medalists vs non-medalists")
ax[1].set_ylabel("Average Games participations")
plt.tight_layout()
plt.show()

## Step 5: Preprocessing pipeline

Numbers: fill missing values with the median, then scale them (needed for Elastic Net, Huber, Bayesian Ridge and the neural network).
Category (season): one-hot encoding.
Everything is inside a `Pipeline`, so in every fold the preprocessing is learned **only from the training part**. This avoids data leakage.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
])
print("Preprocessing pipeline is ready.")

## Step 6: Choose the 5 algorithms

In [ ]:
algorithms = {
    "Elastic Net": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "Bayesian Ridge": BayesianRidge(),
    "Huber Regressor": HuberRegressor(max_iter=300),
    "Bagging Regressor": BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=8), n_estimators=50, random_state=42, n_jobs=-1),
    "MLP Neural Network": MLPRegressor(
        hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=200, random_state=42),
}

print("Algorithms to compare:")
for number, name in enumerate(algorithms, start=1):
    print(f"  {number}. {name}")

## Step 7: 10-Fold Cross-Validation on the FULL dataset

In [ ]:
N_FOLDS = 10
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_rmse = {name: [] for name in algorithms}
train_time = {name: 0.0 for name in algorithms}

progress = tqdm(total=len(algorithms) * N_FOLDS, desc="Cross-validation", unit="fold")

for name, model in algorithms.items():
    for fold_number, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipe = Pipeline([("prep", clone(preprocessor)), ("model", clone(model))])

        start = time.time()
        pipe.fit(X_train, y_train)
        train_time[name] += time.time() - start

        predictions = pipe.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        fold_rmse[name].append(rmse)

        progress.set_postfix(model=name, fold=fold_number)
        progress.update(1)

progress.close()
print("Cross-validation finished.")

## Step 8: Results table

In [ ]:
# For comparison: a "model" that always predicts the average number of participations
baseline_rmse = float(np.std(y))
print(f"Baseline RMSE (always predict the average): {baseline_rmse:.3f}")

results = pd.DataFrame({
    "Algorithm": list(algorithms.keys()),
    "Mean RMSE": [np.mean(fold_rmse[n]) for n in algorithms],
    "Std of RMSE": [np.std(fold_rmse[n]) for n in algorithms],
    "Best Fold RMSE": [np.min(fold_rmse[n]) for n in algorithms],
    "Worst Fold RMSE": [np.max(fold_rmse[n]) for n in algorithms],
    "Total Train Time (s)": [train_time[n] for n in algorithms],
}).sort_values("Mean RMSE").reset_index(drop=True)
results.index = results.index + 1
results.index.name = "Rank"
results

In [ ]:
fold_table = pd.DataFrame(fold_rmse, index=[f"Fold {i}" for i in range(1, N_FOLDS + 1)])
fold_table

## Step 9: Charts

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ordered = results.sort_values("Mean RMSE")
ax[0].bar(ordered["Algorithm"], ordered["Mean RMSE"], yerr=ordered["Std of RMSE"],
          capsize=5, color="#4C78A8")
ax[0].axhline(baseline_rmse, color="red", linestyle="--", label="Baseline (predict average)")
ax[0].set_title("Mean RMSE of each algorithm (10-fold CV)")
ax[0].set_ylabel("RMSE, lower is better")
ax[0].tick_params(axis="x", rotation=30)
ax[0].legend()
for i, v in enumerate(ordered["Mean RMSE"]):
    ax[0].text(i, v + 0.005, f"{v:.3f}", ha="center")

ax[1].boxplot([fold_rmse[n] for n in algorithms], labels=list(algorithms.keys()))
ax[1].set_title("RMSE across the 10 folds")
ax[1].set_ylabel("RMSE")
ax[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(results["Algorithm"], results["Total Train Time (s)"], color="#F58518")
plt.title("Total training time over 10 folds")
plt.ylabel("Seconds")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Step 10: Train the best model on all the data and save it

In [ ]:
best_name = results.iloc[0]["Algorithm"]
print("Best algorithm (lowest mean RMSE):", best_name)

final_model = Pipeline([("prep", clone(preprocessor)), ("model", clone(algorithms[best_name]))])
final_model.fit(X, y)

bundle = {
    "model": final_model,
    "model_name": best_name,
    "cv_rmse": float(results.iloc[0]["Mean RMSE"]),
    "baseline_rmse": baseline_rmse,
    "feature_columns": categorical_cols + numeric_cols,
    "sklearn_version": sklearn.__version__,
}

MODEL_FILE = "olympic_participation_model.pkl"
with open(MODEL_FILE, "wb") as f:
    pickle.dump(bundle, f)
print(f"Saved '{MODEL_FILE}' ({os.path.getsize(MODEL_FILE) / 1e6:.2f} MB)")

In [ ]:
# Quick test: reload the saved file and predict one example athlete
with open(MODEL_FILE, "rb") as f:
    loaded = pickle.load(f)

example = pd.DataFrame([{
    "season": "Summer", "first_game_year": 2008, "birth_year": 1985,
    "age_at_first_game": 23, "gold": 1, "silver": 0, "bronze": 1, "has_bio": 1,
}])[loaded["feature_columns"]]
print(f"Predicted Games participations: {loaded['model'].predict(example)[0]:.2f}")

In [ ]:
# Download the model file to your computer (works in Google Colab)
if IN_COLAB:
    files.download(MODEL_FILE)

## Step 11: Conclusion

*(Write 4 to 6 lines in your own words after you see your results. Ideas:)*
* Which algorithm had the lowest RMSE and which had the highest?
* How much better is the best model than the baseline (always predicting the average)?
* Why is the error still fairly high? (Most athletes have only 1 participation, and the dataset has very few columns, so there is limited information to predict from.)
* How do training times compare? (The neural network and Bagging take longer than the linear models.)
* Which model was deployed and where.